#Lab 6: XGBoost on the Wine Dataset

This notebook uses dataset: `sklearn.datasets.load_wine`.

Goal: train an XGBoost multiclass classifier to identify wine cultivars from chemical measurements.

In [ ]:
import numpy as np
import wandb
import xgboost as xgb
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

## Load a Different Dataset

No external file download is needed.

In [ ]:
dataset = load_wine()

X = dataset.data
y = dataset.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", dataset.target_names)
print("First five features:", dataset.feature_names[:5])

## Train/Test Split

A stratified split keeps the class proportions similar in train and test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=dataset.feature_names)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=dataset.feature_names)

## Configure XGBoost and W&B

Set `WANDB_MODE` to `"online"` after running `wandb login`. Keep it as `"disabled"` for a quick local run.

In [ ]:
WANDB_MODE = "disabled"  # change to "online" or "offline" when you want W&B logging

params = {
    "objective": "multi:softmax",
    "eta": 0.1,
    "max_depth": 4,
    "num_class": len(dataset.target_names),
    "eval_metric": "mlogloss",
    "nthread": 4,
    "seed": 42,
}

num_round = 20
callbacks = []
run = None

if WANDB_MODE != "disabled":
    run = wandb.init(
        project="Lab1-alternative-data",
        name="xgboost-wine-notebook",
        mode=WANDB_MODE,
        config={
            **params,
            "dataset": "sklearn.load_wine",
            "features": list(dataset.feature_names),
            "target_names": list(dataset.target_names),
            "rounds": num_round,
        },
    )
    callbacks.append(wandb.xgboost.WandbCallback(log_model=True))

## Train the Model

In [ ]:
model = xgb.train(
    params,
    dtrain,
    num_round,
    evals=[(dtrain, "train"), (dtest, "test")],
    callbacks=callbacks,
)

## Evaluate the Model

In [ ]:
predictions = model.predict(dtest).astype(int)
accuracy = accuracy_score(y_test, predictions)
error_rate = np.mean(predictions != y_test)

print(f"Test accuracy = {accuracy:.4f}")
print(f"Test error = {error_rate:.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

if run is not None:
    run.summary["Accuracy"] = float(accuracy)
    run.summary["Error Rate"] = float(error_rate)
    wandb.sklearn.plot_confusion_matrix(
        y_test,
        predictions,
        labels=list(dataset.target_names),
    )
    run.finish()

## Assignment Extension

Try at least two more runs with changed hyperparameters, then compare results.

Examples:

- Increase `num_round` from `20` to `50`.
- Change `max_depth` from `4` to `2` or `6`.
- Change `eta` from `0.1` to `0.05` or `0.2`.

Conclusion prompt: which setting gave the best test accuracy, and did any setting look like it was overfitting?